In [ ]:
from pathlib import Path
import os

base_folder = Path("/home/automl/git/iot-threat-classifier/2025-11-17/Input")

dispatcher_filename = Path(os.path.join(base_folder, "start"))

In [ ]:
from datetime import datetime
from time import sleep

def now():
    now = datetime.now()
    yyyymmdd_hhmmss_part = now.strftime('%Y-%m-%d %H:%M:%S')
    ms_part = f'{int(now.microsecond / 1000):03d}'
    return f'{yyyymmdd_hhmmss_part},{ms_part}'

# while not dispatcher_filename.exists():
#     print(f'[{now()}] Dispatcher file does not exist; sleeping...')
#     sleep(300)

print(f'[{now()}] Dispatcher file EXISTS; starting...')

In [ ]:
import json
import numpy as np
import pandas as pd

def load_results(results_filename):
    df = pd.read_excel(results_filename)
    try:
        row = df.loc[df['f1_score_abs'].idxmax(), ['Unnamed: 0', 'f1_score_abs']]
        max_config = str(row['Unnamed: 0'])
        max_f1_weighted = f"{float(row['f1_score_abs']):.6f}"
    except Exception:
        max_config = None
        max_f1_weighted = str(np.nan)
    cfg_str = f"{max_config if max_config else 'N/A':<8}"
    f1_str = f"{max_f1_weighted if max_f1_weighted not in ['nan', None] else 'N/A':<8}"
    return cfg_str, f1_str

def parse_exception(e):
    try:
        return str(e).split("\n")[-2]
    except:
        return "unknown error"

In [ ]:
BINARIZE_FLAGS = [False]
SEEDS = [17, 23, 37, 53, 89]
SAMPLE_FRACS = [0.05, 0.1, 0.2] # do not put "full" here!

In [ ]:
import itertools
from pathlib import Path
from pprint import pprint

def get_dir_size(path):
    root = Path(path)
    return sum(f.stat().st_size for f in root.rglob('*') if f.is_file())

dataset_info = {}

# Find all instances of metadata.json recursively
found_files = base_folder.rglob('metadata.json')

for file_path in found_files:
    with open(file_path, mode='r', encoding='utf-8') as fp:
        metadata = json.load(fp)

    dataset_name = metadata['name']
    dataset_folder_path = os.path.join(base_folder, dataset_name)
    dataset_folder_size = get_dir_size(dataset_folder_path)
    dataset_config = f"{metadata['binarize']};{metadata['seed']};{metadata['sample_frac']}"

    if dataset_name not in dataset_info.keys():
        dataset_info[dataset_name] = {
            'path': dataset_folder_path,
            'size': dataset_folder_size,
            'configs': set()
        }
    dataset_info[dataset_name]['configs'].add(dataset_config)

# Create a new dictionary sorted by size (ascending)
sorted_dataset_info = dict(sorted(
    dataset_info.items(), 
    key=lambda item: item[1]['size']
))

# Create all combinations once
all_configs = {f"{b};{s};{f}" for b, s, f in itertools.product(BINARIZE_FLAGS, SEEDS, SAMPLE_FRACS)}

for k, v in sorted_dataset_info.items():
    # Check if 'all_configs' is a subset of the current dataset's configs
    if all_configs.issubset(v['configs']):
        v['status'] = 'ready'
        print(f"{k} ({v['size']/1024**2:.2f} MB) => ready\n")
    else:
        # Find what configuration is missing
        missing = all_configs - set(v['configs'])
        v['status'] = 'missing'
        print(f"{k} ({v['size']/1024**2:.2f} MB) => missing:\n{missing}\n")

In [ ]:
ready_datasets = [(k, v['path']) for k, v in sorted_dataset_info.items() if v['status'] == 'ready']

ready_datasets

In [ ]:
from tqdm.auto import tqdm
import papermill as pm
import itertools

# 1. Master Job List
job_queue = list(itertools.product(
    ready_datasets,     # [(name, path), (name, path)...]
    BINARIZE_FLAGS, 
    SEEDS,
    SAMPLE_FRACS
))

errors = {}

# 2. Progress Bar
pbar = tqdm(job_queue, desc='Progress')

for (ds_name, ds_input_path), binarize, seed, sample_frac in pbar:
    
    # Description to know what is running
    # pbar.set_description(f"{ds_name} (S={seed} | F={sample_frac} | S={seed})")
    pbar.set_postfix({"DS": ds_name, "B": str(binarize), "S": seed, "F": sample_frac})

    # Path construction
    seed_part = f"seed_{seed}"
    frac_part = f"sampled_{int(100 * sample_frac):02}"
    ds_output_path = ds_input_path.replace(f'/Input/{ds_name}', f'/Output/{ds_name}/{seed_part}/{frac_part}')
    os.makedirs(ds_output_path, exist_ok=True)

    input_notebook = 'evaluator_code_v2.ipynb'
    output_notebook = os.path.join(ds_output_path, 'xgb_execution.ipynb')
    results_filename = os.path.join(ds_output_path, 'xgb_summary_table.xlsx')
    params_file = output_notebook.replace('.ipynb', '_params.json')
    error_file = output_notebook.replace('.ipynb', '_errors.json')

    # Initialize parameters to None to avoid error handler from crashing
    parameters = None

    try:
        # Skip if already done
        if Path(results_filename).exists():
            continue

        parameters = dict(
            dataset_name=ds_name,
            binarize=binarize,
            random_state=seed,
            sample_frac=sample_frac,
            target_column='label',
            min_samples_per_class=1,
            feature_selection_threshold=0.95,
            sample_filtering_quantile=0.10,
            hpo_n_trials=10, 
            hpo_timeout=60, 
            num_boost_round=100,
            early_stopping_rounds=10,
            plot_param_importances=False,
            n_jobs=-1
        )

        # Save params before execution to debug input if it crashes
        with open(params_file, 'w', encoding='utf-8') as f:
            json.dump(parameters, f, indent=4)                
    
        # Execute the job
        pm.execute_notebook(
            input_notebook, 
            output_notebook, 
            parameters=parameters,
            kernel_name='python3',
            log_output=False
        )
        
        # Load results
        # # max_config, max_f1_weighted = load_results(results_filename)
        # tqdm.write(f'[{now()}] SUCCESS | DS = {ds_name} | B = {binarize} | F = {sample_frac} | S = {seed} | F1: {max_f1_weighted:.4f}')
        tqdm.write(f'[{now()}] SUCCESS | DS = {ds_name} | B = {binarize} | S = {seed} | F = {sample_frac} |')

    except Exception as e:

        print(e)
        
        tqdm.write(f'[{now()}] ERROR   | DS = {ds_name} | B = {binarize} | S = {seed} | F = {sample_frac} | {parse_exception(e)}')
        
        error_json = {
            "timestamp": now(), 
            "parameters": parameters if parameters else "Not generated", 
            "error": str(e).split('\n')
        }
        
        with open(error_file, 'w', encoding='utf-8') as f:
            json.dump(error_json, f, indent=4)